# S3 Integration & Batch Pipelines

The database is ready. Now we need to fill it. The photo app's ingestion pipeline has a straightforward job: scan an S3 bucket for image files, download each one, extract its EXIF metadata, and upsert a record into Postgres. Run it twice and nothing should change — the pipeline must be **idempotent**. Run it concurrently across hundreds of files and the throughput should scale linearly — it must be **concurrent**.

In this notebook we build this pipeline from the ground up: a `boto3` S3 client, a paginated listing generator, EXIF extraction with Pillow, idempotent upserts with SQLAlchemy's `on_conflict_do_update`, an `asyncio` worker pool capped by a semaphore, and progress tracking exposed through a FastAPI status endpoint. We also cover presigned URLs — the mechanism for serving photos from a private bucket without making objects public.

## S3 Fundamentals

**Object storage.** Amazon S3 is an **object store** — a flat key-value system where each object is addressed by a bucket name and an arbitrary string key. There are no real directories; the appearance of a folder hierarchy (`photos/2024/beach.jpg`) is a prefix illusion — the key literally contains the slashes. This is fundamentally different from (1) **block storage** (EBS, local disks), where a filesystem sits on top of fixed-size blocks, and (2) **file storage** (EFS, NFS), where a POSIX filesystem is served over a network. Object storage trades filesystem semantics for massive horizontal scalability and eleven nines of durability.

Key concepts:
- **Bucket**: a top-level namespace, globally unique within a region
- **Key**: the full path of an object within a bucket (e.g., `uploads/2024/01/img_001.jpg`)
- **Prefix**: any string that serves as a path filter for listing (`uploads/2024/01/`)
- **Metadata**: per-object key/value pairs stored alongside the object (up to 2 KB)
- **Storage class**: tiered pricing for access frequency: `STANDARD`, `STANDARD_IA`, `GLACIER`, etc.

**`boto3` client vs. resource API.** `boto3` exposes two APIs for each AWS service: a low-level **client** (`boto3.client("s3")`) that maps directly to the AWS REST API, and a higher-level **resource** API that wraps objects as Python classes. We use the client: it is more explicit, better supported, and maps cleanly to the underlying S3 semantics.

**Authentication.** `boto3` follows a **credential chain**: it searches for credentials in this order:
1. Environment variables (`AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_SESSION_TOKEN`)
2. Shared credentials file (`~/.aws/credentials`)
3. AWS config file (`~/.aws/config`)
4. IAM role attached to the compute environment (EC2, Lambda, ECS)

We never hardcode credentials in source code. In development, environment variables loaded via `pydantic-settings` are the cleanest approach.

Installing packages:

```bash
uv add boto3 Pillow piexif
```

Creating a `boto3` S3 client from typed settings:

In [ ]:
from unittest.mock import MagicMock, patch

import boto3
from pydantic_settings import BaseSettings, SettingsConfigDict


class S3Settings(BaseSettings):
    model_config = SettingsConfigDict(env_file=".env", env_file_encoding="utf-8")

    aws_access_key_id: str = "test-key"
    aws_secret_access_key: str = "test-secret"
    aws_region: str = "us-east-1"
    s3_bucket: str = "my-photos-bucket"


s3_settings = S3Settings()

s3 = boto3.client(
    "s3",
    region_name=s3_settings.aws_region,
    aws_access_key_id=s3_settings.aws_access_key_id,
    aws_secret_access_key=s3_settings.aws_secret_access_key,
)

# Mock list_buckets since we have no real AWS credentials in this environment
with patch.object(s3, "list_buckets", return_value={"Buckets": [{"Name": "my-photos-bucket"}]}):
    response = s3.list_buckets()
    print("Buckets:", [b["Name"] for b in response["Buckets"]])

## Paginated Listing

A single `s3.list_objects_v2` call returns at most $1000$ objects. For a bucket with millions of photos we must page through results using the `ContinuationToken` returned in the response. `boto3` paginators handle this automatically: they issue repeated API calls and surface a clean iterator over all pages.

We filter by common photo extensions and yield `s3_key` strings, making the generator composable with the downstream pipeline stages.

Defining the `list_photos` generator using a `boto3` paginator:

In [ ]:
from collections.abc import Generator

PHOTO_EXTENSIONS = {".jpg", ".jpeg", ".png", ".heic", ".dng"}


def list_photos(
    s3_client,
    bucket: str,
    prefix: str = "",
) -> Generator[str, None, None]:
    """Yield S3 keys for all photo objects under `prefix` in `bucket`."""
    paginator = s3_client.get_paginator("list_objects_v2")
    pages = paginator.paginate(Bucket=bucket, Prefix=prefix)

    for page in pages:
        for obj in page.get("Contents", []):
            key: str = obj["Key"]
            ext = "." + key.rsplit(".", 1)[-1].lower() if "." in key else ""
            if ext in PHOTO_EXTENSIONS:
                yield key

Testing `list_photos` against a mocked paginator:

In [ ]:
from unittest.mock import MagicMock

mock_s3 = MagicMock()
mock_paginator = MagicMock()
mock_s3.get_paginator.return_value = mock_paginator
mock_paginator.paginate.return_value = [
    {
        "Contents": [
            {"Key": "photos/2024/beach.jpg"},
            {"Key": "photos/2024/sunset.heic"},
            {"Key": "photos/2024/README.txt"},   # filtered out
            {"Key": "photos/2024/raw.dng"},
        ]
    },
    {
        "Contents": [
            {"Key": "photos/2024/portrait.png"},
        ]
    },
]

keys = list(list_photos(mock_s3, "my-photos-bucket", prefix="photos/2024/"))
print(f"Found {len(keys)} photo(s):")
for k in keys:
    print(f"  {k}")

Estimating pipeline runtime from object count:

In [ ]:
N_PHOTOS = 50_000
SECONDS_PER_PHOTO_SINGLE = 0.8   # download + EXIF + upsert, serial
N_WORKERS = 8
SECONDS_PER_PHOTO_CONCURRENT = SECONDS_PER_PHOTO_SINGLE / N_WORKERS

serial_hours = N_PHOTOS * SECONDS_PER_PHOTO_SINGLE / 3600
concurrent_minutes = N_PHOTOS * SECONDS_PER_PHOTO_CONCURRENT / 60

print(f"Serial ({N_PHOTOS:,} photos): {serial_hours:.1f} h")
print(f"Concurrent (N={N_WORKERS}):  {concurrent_minutes:.1f} min")

## EXIF Extraction

**EXIF** (Exchangeable Image File Format) is a metadata standard embedded in JPEG, TIFF, and HEIC image files. It records the camera settings, timestamp, and GPS coordinates at the moment of capture. The fields most useful to the photo app are:

| EXIF Tag | Field | Example |
|----------|-------|--------|
| `DateTimeOriginal` | Capture timestamp | `"2024:07:15 14:32:01"` |
| `GPSInfo` | GPS fix (lat/lon/alt) | rational tuples |
| `Make` / `Model` | Camera manufacturer and model | `"Apple"` / `"iPhone 15 Pro"` |
| `ExposureTime` | Shutter speed as a fraction | `(1, 250)` → $1/250$ s |
| `FNumber` | Aperture f-stop | `(28, 10)` → $f/2.8$ |
| `ISOSpeedRatings` | ISO sensitivity | `125` |
| `ImageWidth` / `ImageHeight` | Pixel dimensions | `4032` / `3024` |

GPS coordinates are stored as **rational numbers**: tuples of `(numerator, denominator)` representing degrees, minutes, and seconds. We convert them to decimal degrees, which is the standard format for geographic coordinates:

$$\text{decimal} = d + \frac{m}{60} + \frac{s}{3600}$$

where $d$, $m$, $s$ are degrees, minutes, and seconds respectively.

**NOTE:** HEIC files from iPhones require `pyheif` or `pillow-heif` for decoding; `ExifTool` (a command-line utility) handles nearly every RAW and HEIC variant and is the most reliable option for production. We use Pillow here since it handles JPEG and PNG without additional system dependencies.

Defining the `extract_exif` function:

In [ ]:
import io
from typing import Any

from PIL import Image
from PIL.ExifTags import TAGS, GPSTAGS


def _rational_to_float(rational) -> float:
    """Convert a Pillow IFDRational or (num, denom) tuple to float."""
    if hasattr(rational, "numerator"):
        return rational.numerator / rational.denominator if rational.denominator else 0.0
    num, denom = rational
    return num / denom if denom else 0.0


def _gps_to_decimal(gps_info: dict) -> tuple[float, float] | None:
    """Convert EXIF GPS info to (latitude, longitude) in decimal degrees."""
    try:
        lat_vals = gps_info[2]   # GPSLatitude
        lat_ref = gps_info[1]    # GPSLatitudeRef ('N' or 'S')
        lon_vals = gps_info[4]   # GPSLongitude
        lon_ref = gps_info[3]    # GPSLongitudeRef ('E' or 'W')

        lat = (_rational_to_float(lat_vals[0])
               + _rational_to_float(lat_vals[1]) / 60
               + _rational_to_float(lat_vals[2]) / 3600)
        lon = (_rational_to_float(lon_vals[0])
               + _rational_to_float(lon_vals[1]) / 60
               + _rational_to_float(lon_vals[2]) / 3600)

        if lat_ref == "S":
            lat = -lat
        if lon_ref == "W":
            lon = -lon

        return lat, lon
    except (KeyError, TypeError, ZeroDivisionError):
        return None


def extract_exif(image_bytes: bytes) -> dict[str, Any]:
    """Extract EXIF metadata from a JPEG/PNG image as a plain dict."""
    try:
        img = Image.open(io.BytesIO(image_bytes))
        raw_exif = img._getexif()  # returns tag_id -> value dict or None
    except Exception:
        return {}

    if raw_exif is None:
        return {}

    exif: dict[str, Any] = {}
    for tag_id, value in raw_exif.items():
        tag_name = TAGS.get(tag_id, str(tag_id))
        if tag_name == "GPSInfo":
            coords = _gps_to_decimal(value)
            if coords:
                exif["latitude"], exif["longitude"] = coords
        elif tag_name in {"Make", "Model", "DateTimeOriginal", "ISOSpeedRatings",
                          "ImageWidth", "ImageLength"}:
            exif[tag_name] = str(value) if not isinstance(value, (int, float, str)) else value
        elif tag_name in {"ExposureTime", "FNumber"}:
            try:
                exif[tag_name] = _rational_to_float(value)
            except (TypeError, AttributeError):
                pass

    return exif

Testing `extract_exif` on a synthetic JPEG with known metadata:

In [ ]:
import io
import struct

import piexif
from PIL import Image


def make_test_jpeg() -> bytes:
    """Create a minimal JPEG with DateTimeOriginal and Make EXIF tags."""
    img = Image.new("RGB", (640, 480), color=(100, 149, 237))

    exif_dict = {
        "0th": {
            piexif.ImageIFD.Make: b"Apple",
            piexif.ImageIFD.Model: b"iPhone 15 Pro",
        },
        "Exif": {
            piexif.ExifIFD.DateTimeOriginal: b"2024:07:15 14:32:01",
            piexif.ExifIFD.ISOSpeedRatings: 125,
            piexif.ExifIFD.FNumber: (28, 10),
            piexif.ExifIFD.ExposureTime: (1, 250),
        },
    }
    exif_bytes = piexif.dump(exif_dict)

    buf = io.BytesIO()
    img.save(buf, format="JPEG", exif=exif_bytes)
    return buf.getvalue()


jpeg_bytes = make_test_jpeg()
exif_data = extract_exif(jpeg_bytes)

print("Extracted EXIF:")
for k, v in exif_data.items():
    print(f"  {k}: {v}")

Verifying that missing EXIF returns an empty dict gracefully:

In [ ]:
img_no_exif = Image.new("RGB", (100, 100), color=(255, 255, 255))
buf = io.BytesIO()
img_no_exif.save(buf, format="JPEG")
result = extract_exif(buf.getvalue())
print(f"EXIF from plain JPEG: {result!r}")

## Idempotent Upserts

**Idempotency** means that running the pipeline any number of times (twice, or a hundred times) produces exactly the same database state as running it once. Without it, restarting after a failure would create duplicate records. The natural key for idempotency is `s3_key`: each photo has exactly one S3 object path, so we can use it as a conflict target.

PostgreSQL's `INSERT ... ON CONFLICT (s3_key) DO UPDATE SET ...` implements upsert semantics: if a row with the same `s3_key` already exists, update its mutable fields (`exif`, `indexed_at`); otherwise insert a fresh row. SQLAlchemy exposes this through `insert().on_conflict_do_update()`:

```sql
INSERT INTO photos (s3_key, filename, exif, indexed_at, created_at)
VALUES ($1, $2, $3, $4, now())
ON CONFLICT (s3_key)
DO UPDATE SET
    exif = EXCLUDED.exif,
    indexed_at = EXCLUDED.indexed_at;
```

`EXCLUDED` refers to the row that *would have been inserted*: the proposed new values. This pattern is safe to call on every pipeline run regardless of whether the photo has been seen before.

Defining the `upsert_photo` function using SQLAlchemy's dialect-specific insert:

In [ ]:
from datetime import datetime, timezone
from typing import Any

from sqlalchemy.dialects.postgresql import insert as pg_insert
from sqlalchemy.ext.asyncio import AsyncSession


async def upsert_photo(
    session: AsyncSession,
    s3_key: str,
    filename: str,
    exif: dict[str, Any],
    width: int | None = None,
    height: int | None = None,
) -> None:
    """Insert or update a photo row; idempotent on s3_key."""
    now = datetime.now(tz=timezone.utc)
    taken_at_str = exif.get("DateTimeOriginal")
    taken_at = None
    if taken_at_str:
        try:
            taken_at = datetime.strptime(taken_at_str, "%Y:%m:%d %H:%M:%S").replace(
                tzinfo=timezone.utc
            )
        except ValueError:
            pass

    stmt = (
        pg_insert(Photo)                                             # <1>
        .values(
            s3_key=s3_key,
            filename=filename,
            taken_at=taken_at,
            width=width,
            height=height,
            exif=exif,
            indexed_at=now,
        )
        .on_conflict_do_update(                                      # <2>
            index_elements=["s3_key"],
            set_={
                "exif": exif,
                "indexed_at": now,
                "taken_at": taken_at,
                "width": width,
                "height": height,
            },
        )
    )
    await session.execute(stmt)


print("upsert_photo defined")

1. We use `sqlalchemy.dialects.postgresql.insert` (not the generic `sqlalchemy.insert`) to access the `.on_conflict_do_update()` method, which is a PostgreSQL extension.
2. `index_elements=["s3_key"]` tells Postgres which conflict target to use; this must correspond to a UNIQUE constraint or index on the table.

## Concurrent Workers

The ingestion pipeline is **I/O-bound**: each job downloads from S3, extracts EXIF, and writes to Postgres. All three operations spend most of their time waiting for network or disk responses rather than running CPU instructions. `asyncio` concurrency is the right tool: we can issue many downloads simultaneously without spinning up OS threads or processes, keeping memory overhead low.

**Design.** We use an `asyncio.Queue` as the work buffer: the main coroutine enqueues all S3 keys, and $N$ worker coroutines pull from the queue until it is empty. An `asyncio.Semaphore` caps the number of concurrent S3 downloads to avoid throttling or overwhelming the connection pool.

The worker pool pattern:

```
[list_photos] → Queue → [worker-0]  → upsert_photo
                         [worker-1]  → upsert_photo
                         ...         ...
                         [worker-N]  → upsert_photo
```

Defining the async pipeline with a semaphore-capped worker pool:

In [ ]:
import asyncio
import time
from collections import Counter


async def download_photo(s3_key: str, delay: float = 0.1) -> bytes:
    """Simulate an S3 download with a configurable async sleep."""
    await asyncio.sleep(delay)   # stand-in for s3.get_object(...)
    return b"fake-image-bytes"


async def process_key(
    s3_key: str,
    sem: asyncio.Semaphore,
    stats: Counter,
) -> None:
    """Download, extract EXIF, and upsert one photo."""
    async with sem:
        try:
            image_bytes = await download_photo(s3_key)
            exif = extract_exif(image_bytes)         # sync; fast enough in-process
            # await upsert_photo(session, s3_key, ...)  # omitted: no live DB here
            stats["processed"] += 1
        except Exception as exc:
            stats["failed"] += 1
            print(f"  ERROR {s3_key}: {exc}")


async def run_pipeline(
    keys: list[str],
    n_workers: int = 8,
    max_concurrent: int = 8,
) -> Counter:
    sem = asyncio.Semaphore(max_concurrent)
    stats: Counter = Counter()
    await asyncio.gather(*[process_key(k, sem, stats) for k in keys])
    return stats


print("Pipeline functions defined")

Benchmarking serial vs. concurrent throughput on simulated keys:

In [ ]:
import nest_asyncio
nest_asyncio.apply()

SYNTHETIC_KEYS = [f"photos/2024/img_{i:04d}.jpg" for i in range(40)]
SIMULATED_DELAY = 0.05   # 50 ms per download


async def bench(n_concurrent: int) -> float:
    sem = asyncio.Semaphore(n_concurrent)
    stats: Counter = Counter()
    t0 = time.perf_counter()
    await asyncio.gather(*[process_key(k, sem, stats) for k in SYNTHETIC_KEYS])
    elapsed = time.perf_counter() - t0
    return elapsed


for n in [1, 4, 8, 16]:
    elapsed = asyncio.run(bench(n))
    rate = len(SYNTHETIC_KEYS) / elapsed
    print(f"N={n:2d}  →  {elapsed:.2f}s  ({rate:.1f} photos/s)")

:::{.callout-caution}
Setting `max_concurrent` too high can trigger S3 throttling (`SlowDown` errors with HTTP `503`) or exhaust the SQLAlchemy connection pool. A value of $8$–$16$ is a safe default; increase it only after measuring actual throughput against your specific bucket and database configuration.

:::

## Progress Tracking

A long-running pipeline that produces no output is opaque and hard to operate. We add two layers of progress visibility: (1) a `tqdm` progress bar for interactive runs, and (2) a `pipeline_runs` state object that exposes progress through a FastAPI endpoint.

**`tqdm.asyncio.tqdm`.** The `tqdm` library provides async-aware progress tracking that integrates cleanly with `asyncio.gather`:

```python
from tqdm.asyncio import tqdm

await tqdm.gather(*[process_key(k, sem, stats) for k in keys])
```

**Pipeline run state.** For programmatic access we track the current run in a Pydantic model and expose it through a `GET /pipeline/status` endpoint. In production this state would be persisted to a `pipeline_runs` table in Postgres; here we keep it in memory for simplicity.

Defining the `PipelineRun` model and the status endpoint:

In [ ]:
import uuid
from datetime import datetime, timezone
from typing import Literal, Optional

from fastapi import FastAPI
from pydantic import BaseModel


class PipelineRun(BaseModel):
    id: uuid.UUID = uuid.uuid4()
    bucket: str
    prefix: str = ""
    total: int = 0
    processed: int = 0
    failed: int = 0
    started_at: Optional[datetime] = None
    finished_at: Optional[datetime] = None
    status: Literal["idle", "running", "done", "error"] = "idle"

    @property
    def progress_pct(self) -> float:
        return round(100 * self.processed / self.total, 1) if self.total else 0.0


# In-memory state (replace with DB persistence in production)
current_run: PipelineRun = PipelineRun(bucket="my-photos-bucket")

pipeline_app = FastAPI(title="Pipeline API")


@pipeline_app.get("/pipeline/status", response_model=PipelineRun)
async def get_pipeline_status() -> PipelineRun:
    return current_run


# Simulate a mid-run state
current_run.status = "running"
current_run.total = 50_000
current_run.processed = 12_430
current_run.failed = 3
current_run.started_at = datetime(2026, 4, 5, 9, 0, 0, tzinfo=timezone.utc)

print(current_run.model_dump_json(indent=2))

## Presigned URLs

Photos live in a private S3 bucket, so they are not publicly accessible by default. To serve them to users (or to a frontend), we need a way to grant temporary read access without making the bucket public. **Presigned URLs** are the standard solution: a time-limited, cryptographically signed URL that grants the bearer access to a specific S3 object for a specified duration. No AWS credentials are needed to use the URL — only to generate it.

The URL is generated by signing the `GetObject` request with the AWS credentials:

```python
url = s3.generate_presigned_url(
    "get_object",
    Params={"Bucket": bucket, "Key": s3_key},
    ExpiresIn=3600,   # seconds; 1 hour
)
```

The resulting URL looks like:

```
https://my-photos-bucket.s3.amazonaws.com/photos/beach.jpg
    ?X-Amz-Algorithm=AWS4-HMAC-SHA256
    &X-Amz-Credential=AKID.../20260405/us-east-1/s3/aws4_request
    &X-Amz-Date=20260405T120000Z
    &X-Amz-Expires=3600
    &X-Amz-Signature=abc123...
```

A `GET /photos/{id}/url` endpoint generates and returns the presigned URL on demand. The expiry should be short (1–24 hours is typical), since the URL cannot be revoked once issued.

Defining `generate_presigned_url` and the FastAPI endpoint:

In [ ]:
import uuid
from unittest.mock import MagicMock

from fastapi import APIRouter, HTTPException, status
from pydantic import BaseModel


class PresignedURLResponse(BaseModel):
    photo_id: uuid.UUID
    url: str
    expires_in: int


def generate_presigned_url(
    s3_client,
    bucket: str,
    s3_key: str,
    expires_in: int = 3600,
) -> str:
    """Return a presigned GET URL for an S3 object."""
    return s3_client.generate_presigned_url(
        "get_object",
        Params={"Bucket": bucket, "Key": s3_key},
        ExpiresIn=expires_in,
    )


url_router = APIRouter(prefix="/photos", tags=["photos"])


@url_router.get("/{photo_id}/url", response_model=PresignedURLResponse)
async def get_photo_url(photo_id: uuid.UUID) -> PresignedURLResponse:
    # In production: fetch s3_key from DB via Depends(get_db)
    s3_key = f"photos/{photo_id}.jpg"   # placeholder
    url = generate_presigned_url(s3, s3_settings.s3_bucket, s3_key)
    return PresignedURLResponse(photo_id=photo_id, url=url, expires_in=3600)


# Demonstrate with mocked boto3
mock_s3 = MagicMock()
expected_url = (
    "https://my-photos-bucket.s3.amazonaws.com/photos/beach.jpg"
    "?X-Amz-Algorithm=AWS4-HMAC-SHA256&X-Amz-Expires=3600&X-Amz-Signature=abc123"
)
mock_s3.generate_presigned_url.return_value = expected_url

url = generate_presigned_url(mock_s3, "my-photos-bucket", "photos/beach.jpg", expires_in=3600)
print("Presigned URL:", url[:80], "...")

:::{.callout-important}
Presigned URLs cannot be revoked once issued. Do not generate them with expirations longer than necessary: 1 hour is a reasonable default for photo viewing. For download links you want users to be able to share, consider a separate short-lived token system rather than long-lived presigned URLs.

:::

## Appendix: Handling HEIC and RAW {#sec-heic-raw}

Modern smartphone and camera workflows produce image formats that Pillow alone cannot decode:

**HEIC.** Apple's default format on iOS 11+. It delivers significantly better compression than JPEG at equivalent quality. Pillow does not support HEIC natively; `pillow-heif` (wrapping `libheif`) extends Pillow with HEIC/HEIF support:

```bash
uv add pillow-heif
```

```python
import pillow_heif
pillow_heif.register_heif_opener()   # patches PIL.Image.open to handle HEIC

img = Image.open("photo.heic")       # works after registration
img_rgb = img.convert("RGB")
```

**RAW formats** (`.dng`, `.cr2`, `.nef`). RAW files contain unprocessed sensor data and require a "demosaicing" step to produce an RGB image. `rawpy` wraps `libraw` for this:

```bash
uv add rawpy imageio
```

```python
import rawpy
import imageio

with rawpy.imread("photo.dng") as raw:
    rgb = raw.postprocess()    # numpy array, shape (H, W, 3)

img = Image.fromarray(rgb)
```

**Thumbnail generation.** The pipeline stores full-resolution images in S3 but we want fast-loading previews for the gallery UI. Pillow's `thumbnail` method resizes in-place while preserving aspect ratio:

```python
MAX_SIZE = (800, 800)

img.thumbnail(MAX_SIZE, Image.LANCZOS)

buf = io.BytesIO()
img.save(buf, format="JPEG", quality=85, optimize=True)
thumbnail_bytes = buf.getvalue()

# Upload thumbnail under a separate prefix
s3.put_object(Bucket=bucket, Key=f"thumbnails/{s3_key}", Body=thumbnail_bytes)
```

We upload thumbnails under a `thumbnails/` prefix, keeping the originals untouched. The frontend requests `GET /photos/{id}/url?size=thumb` which resolves to the thumbnail key rather than the original.

---

■